In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
from collections import Counter
from pathlib import Path
import os
from collections import defaultdict
from scoreboard_reader import process_youtube_video
import shutil

In [3]:
# CONFIG
REPO_ROOT = Path(os.getcwd()).parent  


ROBOT_MODEL_PATH = REPO_ROOT / "yolov8_model" / "best_tuned_yolov8.pt"
NUMBER_MODEL_PATH = REPO_ROOT / "number_reading" / "best_number.pt"

ROBOT_CLASS_ID = 1
BLUE_NUMBER_CLASS_ID = 0
RED_NUMBER_CLASS_ID = 1

FRAME_SKIP = 15

# Custom tracker 
CUSTOM_TRACKER_PATH = REPO_ROOT / "trackers" / "botsort_custom.yaml"


In [4]:
# LOAD MODELS
robot_model = YOLO(ROBOT_MODEL_PATH)
number_model = YOLO(NUMBER_MODEL_PATH)

reader = easyocr.Reader(['en'], gpu=True)

In [5]:
# BOTSORT
print("Loading model...")
model = YOLO(ROBOT_MODEL_PATH)

def run_botsort(model_path, video_path, tracker_path, save_video=False, output_dir=None, stride=1):

    print("Running BotSort tracking...")

    results = model.track(
        source=video_path,
        tracker=tracker_path,
        stream=True,
        persist=True,
        conf=0.35,
        device=0,
        vid_stride=stride,
        verbose=False
    )

    tracking_results = []
    video_writer = None

    for frame_i, result in enumerate(results):

        if result.boxes is None or result.boxes.xyxy is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        ids = result.boxes.id

        if ids is None:
            continue

        ids = ids.cpu().numpy().astype(int)

        # Filter only robot class
        robot_indices = [i for i, c in enumerate(classes) if c == ROBOT_CLASS_ID]

        frame = result.plot()

        for i in robot_indices:

            x1, y1, x2, y2 = boxes[i]
            track_id = ids[i]

            # STORE RESULT
            tracking_results.append({
                "frame": frame_i * stride,
                "track_id": int(track_id),
                "box": [float(x1), float(y1), float(x2), float(y2)],
                "crop": frame[int(y1):int(y2), int(x1):int(x2)].copy()
            })

            # DRAW
            cv2.putText(
                frame,
                f"ID {track_id}",
                (int(x1), int(y1) - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

        if save_video:
            if video_writer is None:
                h, w = frame.shape[:2]
                output_path = output_dir / "botsort_with_ids.mp4"

                video_writer = cv2.VideoWriter(
                    str(output_path),
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    30,
                    (w, h)
                )

            video_writer.write(frame)

    if video_writer:
        video_writer.release()

    print("Tracking complete.")
    return tracking_results

Loading model...


In [6]:
# IMAGE PROCESSING FUNCTIONS
def estimate_angle_from_crop(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return 0.0

    cnt = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(cnt)
    angle = rect[-1]

    if angle < -45:
        angle += 90

    return angle


def rotate_image(img, angle):
    h, w = img.shape[:2]
    center = (w // 2, h // 2)

    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(
        img,
        M,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )


def preprocess_crop(crop, threshold):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

    angle = estimate_angle_from_crop(crop)
    rotated = rotate_image(thresh, angle)

    return rotated

def preprocess_variants(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    variants = []

    # 1. Simple thresholds
    for t in [150, 165, 180, 195, 210]:
        _, th = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
        variants.append(th)

    # 2. Adaptive threshold
    adaptive = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )
    variants.append(adaptive)

    # 3. Otsu
    _, otsu = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    variants.append(otsu)

    # 4. Histogram equalization + Otsu
    eq = cv2.equalizeHist(gray)
    _, th_eq = cv2.threshold(eq, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants.append(th_eq)

    return variants

In [7]:
# OCR
def read_number_from_image(img):
    variants = preprocess_variants(img)
    guesses = []

    angle = estimate_angle_from_crop(img)
    for processed in variants:
        rotated = rotate_image(processed, angle)

        results = reader.readtext(
            rotated,
            allowlist="0123456789",
            detail=1,
            paragraph=False
        )

        # Filter low confidence
        results = [r for r in results if r[2] > 0.5]

        if results:
            guesses.append(results[0][1])

    if not guesses:
        return None

    c = Counter(guesses)
    max_count = max(c.values())

    tied = [num for num, count in c.items() if count == max_count]

    return max(tied, key=lambda x: len(str(x)))

In [8]:
# MATCHING SYSTEM
def levenshtein(a, b):
    a, b = str(a), str(b)
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]

    for i in range(len(a)+1):
        dp[i][0] = i
    for j in range(len(b)+1):
        dp[0][j] = j

    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[-1][-1]


def similarity(a, b):
    if not a or not b:
        return 0
    dist = levenshtein(a, b)
    return 1 - dist / max(len(str(a)), len(str(b)))


def alignment_score(a, b):
    a, b = str(a), str(b)
    best = 0

    for shift in range(-len(b), len(a)+1):
        matches = 0
        for i in range(len(a)):
            j = i - shift
            if 0 <= j < len(b) and a[i] == b[j]:
                matches += 1
        best = max(best, matches)

    return best / max(len(a), len(b))


def combined_score(a, b, w1=0.7, w2=0.3):
    return w1 * similarity(a, b) + w2 * alignment_score(a, b)


def match_number_single(detected, nums):
    if detected is None:
        return None

    best_score = 0.5
    match = None

    for n in nums:
        current = combined_score(detected, n)
        if current > best_score:
            best_score = current
            match = n

    return match

In [9]:
def apply_elimination(final_labels, track_candidates, all_ids, tracks):
    assigned_ids = set(final_labels.values())
    remaining_ids = set(all_ids) - assigned_ids

    # Tracks with no label yet
    unlabeled_tracks = [t for t in tracks if t not in final_labels]

    # Case 1: Perfect elimination
    if len(unlabeled_tracks) == len(remaining_ids):
        for t, rid in zip(unlabeled_tracks, remaining_ids):
            final_labels[t] = rid

    # Case 2: Candidate filtering
    else:
        for t in unlabeled_tracks:
            candidates = set(track_candidates.get(t, []))
            candidates -= assigned_ids

            if len(candidates) == 1:
                final_labels[t] = candidates.pop()

    return final_labels

In [10]:
# NUMBER READING
def read_numbers(tracking_results, blue_team_numbers=[], red_team_numbers=[], stride=1):

    results = []
    current_frame = None
    frame_data = []

    for entry in tracking_results:

        frame_idx = entry["frame"]
        robot_crop = entry["crop"]

        if frame_idx % stride != 0:
            continue

        # Detect frame change
        if current_frame is None:
            current_frame = frame_idx

        if frame_idx != current_frame:
            print(f"Frame {current_frame}: {frame_data}")
            results.append({
                "frame": current_frame,
                "detections": frame_data
            })
            frame_data = []
            current_frame = frame_idx

        if robot_crop is None or robot_crop.size == 0:
            continue

        number_results = number_model(robot_crop, verbose=False)[0]

        for nbox in number_results.boxes:
            cls_id = int(nbox.cls[0])
            if cls_id == RED_NUMBER_CLASS_ID:
                team_list = red_team_numbers
            elif cls_id == BLUE_NUMBER_CLASS_ID:
                team_list = blue_team_numbers
            else:
                continue

            if nbox.conf[0] < 0.6:
                continue

            nx1, ny1, nx2, ny2 = map(int, nbox.xyxy[0])

            # Padding
            pad = 5
            h2, w2 = robot_crop.shape[:2]
            nx1 = max(0, nx1 - pad)
            ny1 = max(0, ny1 - pad)
            nx2 = min(w2, nx2 + pad)
            ny2 = min(h2, ny2 + pad)

            # Filter tiny boxes
            if (nx2 - nx1) < 30 or (ny2 - ny1) < 15:
                continue

            number_crop = robot_crop[ny1:ny2, nx1:nx2]

            if number_crop.size == 0:
                continue

            detected = read_number_from_image(number_crop)
            matched = match_number_single(detected, team_list)

            frame_data.append({
                "detected": detected,
                "matched": matched,
                "track_id": entry["track_id"],
                "box": entry["box"],
                "alliance_hint": "blue" if cls_id == BLUE_NUMBER_CLASS_ID else "red"
            })

    # Add last frame
    if frame_data:
        print(f"Frame {current_frame}: {frame_data}")
        results.append({
            "frame": current_frame,
            "detections": frame_data
        })

    return results

In [11]:
def run_full_pipeline(
    video_path,
    robot_model_path,
    number_model_path,
    tracker_path,
    blue_ids,
    red_ids,
    output_path,
    tracking_stride=1,
    ocr_stride=1
):
    print("Running BotSort...")
    botsort_results = run_botsort(
        robot_model_path,
        video_path,
        tracker_path,
        stride=tracking_stride
    )

    print("Reading numbers...")
    output = read_numbers(
        botsort_results,
        blue_ids,
        red_ids,
        stride=ocr_stride
    )

    # -----------------------------
    # STEP 1: STRONG ALLIANCE LOCK
    # -----------------------------
    track_alliance_votes = defaultdict(lambda: {"blue": 0, "red": 0})

    for frame in output:
        for det in frame["detections"]:
            tid = det["track_id"]
            match = det["matched"]

            # Strong signal: detector class
            hint = det.get("alliance_hint")
            if hint == "blue":
                track_alliance_votes[tid]["blue"] += 2  
            elif hint == "red":
                track_alliance_votes[tid]["red"] += 2

            # Weak signal: OCR match
            if match in blue_ids:
                track_alliance_votes[tid]["blue"] += 1
            elif match in red_ids:
                track_alliance_votes[tid]["red"] += 1

    track_alliance = {}
    track_alliance_confidence = {}

    for tid, votes in track_alliance_votes.items():
        total = votes["blue"] + votes["red"]

        if total == 0:
            track_alliance[tid] = None
            track_alliance_confidence[tid] = 0
            continue

        if votes["blue"] > votes["red"]:
            track_alliance[tid] = "blue"
            track_alliance_confidence[tid] = votes["blue"] / total
        elif votes["red"] > votes["blue"]:
            track_alliance[tid] = "red"
            track_alliance_confidence[tid] = votes["red"] / total
        else:
            track_alliance[tid] = None
            track_alliance_confidence[tid] = 0.5

    # -----------------------------
    # STEP 2: STRICT TRACK MEMORY
    # -----------------------------
    track_memory = defaultdict(list)

    for frame in output:
        for det in frame["detections"]:
            tid = det["track_id"]
            match = det["matched"]

            if match is None:
                continue

            alliance = track_alliance.get(tid)

            # HARD FILTER: only accept correct alliance matches
            if alliance == "blue" and match in blue_ids:
                track_memory[tid].append(match)
            elif alliance == "red" and match in red_ids:
                track_memory[tid].append(match)

    # -----------------------------
    # STEP 3: RECENCY-WEIGHTED VOTE
    # -----------------------------

    final_labels = {}

    for tid, nums in track_memory.items():
        if len(nums) < 2:
            continue  # ignore weak tracks

        scores = defaultdict(float)

        # New: exponential decay weighting
        for i, num in enumerate(nums):
            weight = 0.9 ** (len(nums) - i)  # newer = higher weight
            scores[num] += weight

        best = max(scores, key=scores.get)

        # Optional safety check (prevents super noisy flips)
        total_weight = sum(scores.values())
        if scores[best] / total_weight >= 0.5:
            final_labels[tid] = best

    # -----------------------------
    # STEP 4: ELIMINATION (SAFE)
    # -----------------------------
    def eliminate(alliance_tracks, alliance_ids):
        assigned = set(
            v for k, v in final_labels.items()
            if track_alliance.get(k) in alliance_tracks
        )

        remaining_ids = set(alliance_ids) - assigned

        unlabeled_tracks = [
            t for t in track_alliance
            if track_alliance[t] in alliance_tracks and t not in final_labels
        ]

        if len(unlabeled_tracks) == len(remaining_ids):
            for t, rid in zip(unlabeled_tracks, remaining_ids):
                final_labels[t] = rid

    eliminate({"blue"}, blue_ids)
    eliminate({"red"}, red_ids)

    # -----------------------------
    # STEP 5: FINAL SAFETY FILTER
    # -----------------------------
    for tid, label in list(final_labels.items()):
        alliance = track_alliance.get(tid)

        if alliance == "blue" and label not in blue_ids:
            del final_labels[tid]
        elif alliance == "red" and label not in red_ids:
            del final_labels[tid]

    print("Final track labels:", final_labels)

    # -----------------------------
    # STEP 6: BUILD LOOKUP
    # -----------------------------
    tracking_lookup = defaultdict(list)

    for entry in botsort_results:
        tracking_lookup[entry["frame"]].append(entry)

    # -----------------------------
    # STEP 7: DRAW VIDEO
    # -----------------------------
    print("Drawing Video...")

    cap = cv2.VideoCapture(video_path)
    video_writer = None

    frame_idx = 0
    last_drawn = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        if frame_idx in tracking_lookup:
            draw_frame = frame.copy()

            frame_tracks = tracking_lookup[frame_idx]

            blue_tracks_in_frame = []
            red_tracks_in_frame = []

            for entry in frame_tracks:
                tid = entry["track_id"]
                alliance = track_alliance.get(tid)

                if alliance == "blue":
                    blue_tracks_in_frame.append(tid)
                elif alliance == "red":
                    red_tracks_in_frame.append(tid)

            # --- BLUE elimination ---
            blue_labeled = {
                tid: final_labels[tid]
                for tid in blue_tracks_in_frame
                if tid in final_labels
            }

            blue_unlabeled = [
                tid for tid in blue_tracks_in_frame
                if tid not in final_labels
            ]

            remaining_blue_ids = set(blue_ids) - set(blue_labeled.values())

            if len(blue_unlabeled) == 1 and len(remaining_blue_ids) == 1:
                final_labels[blue_unlabeled[0]] = list(remaining_blue_ids)[0]

            # --- RED elimination ---
            red_labeled = {
                tid: final_labels[tid]
                for tid in red_tracks_in_frame
                if tid in final_labels
            }

            red_unlabeled = [
                tid for tid in red_tracks_in_frame
                if tid not in final_labels
            ]

            remaining_red_ids = set(red_ids) - set(red_labeled.values())

            if len(red_unlabeled) == 1 and len(remaining_red_ids) == 1:
                final_labels[red_unlabeled[0]] = list(remaining_red_ids)[0]


            for entry in tracking_lookup[frame_idx]:
                tid = entry["track_id"]
                x1, y1, x2, y2 = map(int, entry["box"])

                # Determine color based on alliance
                alliance = track_alliance.get(tid)

                if alliance == "blue":
                    color = (255, 0, 0)   # Blue (BGR)
                elif alliance == "red":
                    color = (0, 0, 255)   # Red (BGR)
                else:
                    color = (200, 200, 200)  # Unknown = gray

                # Draw box
                cv2.rectangle(draw_frame, (x1, y1), (x2, y2), color, 2)

                # Label text
                if tid in final_labels:
                    text = str(final_labels[tid])
                else:
                    text = f"ID {tid}"

                # Draw text
                cv2.putText(
                    draw_frame,
                    text,
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2
                )

            last_drawn = draw_frame

        if last_drawn is None:
            continue

        if video_writer is None:
            h, w = last_drawn.shape[:2]
            video_writer = cv2.VideoWriter(
                str(output_path),
                cv2.VideoWriter_fourcc(*"mp4v"),
                30,
                (w, h)
            )

        video_writer.write(last_drawn)

    cap.release()

    if video_writer:
        video_writer.release()

    print("Pipeline complete. Video saved to:", output_path)

    return {
        "tracking": botsort_results,
        "ocr_output": output,
        "final_labels": final_labels
    }

In [28]:
df = process_youtube_video("https://www.youtube.com/watch?v=NSWVoO4ZDEs")

[youtube] Extracting URL: https://www.youtube.com/watch?v=NSWVoO4ZDEs
[youtube] NSWVoO4ZDEs: Downloading webpage


[youtube] NSWVoO4ZDEs: Downloading android vr player API JSON
[info] NSWVoO4ZDEs: Downloading 1 format(s): 137+251
[download] Destination: videos/Final 1 - 2025 Rocket City Regional.f137.mp4
[download] 100% of   91.87MiB in 00:00:03 at 26.54MiB/s    
[download] Destination: videos/Final 1 - 2025 Rocket City Regional.f251.webm
[download] 100% of    2.30MiB in 00:00:00 at 14.33MiB/s  
[Merger] Merging formats into "videos/Final 1 - 2025 Rocket City Regional.mp4"
Deleting original file videos/Final 1 - 2025 Rocket City Regional.f251.webm (pass -k to keep)
Deleting original file videos/Final 1 - 2025 Rocket City Regional.f137.mp4 (pass -k to keep)

0: 384x640 (no detections), 7.0ms
Speed: 1.2ms preprocess, 7.0ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 7.5ms
Speed: 1.7ms preprocess, 7.5ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 7.2ms
Speed: 1.6ms preprocess, 7.2ms inference, 0.4ms pos

In [29]:
from collections import Counter

counts = Counter(team for teams in df["blue_team_numbers"] for team in teams)
majority = counts.most_common(3)
BLUE_IDS = [majority[0][0], majority[1][0], majority[2][0]]
counts = Counter(team for teams in df["red_team_numbers"] for team in teams)
majority = counts.most_common(3)
RED_IDS = [majority[0][0], majority[1][0], majority[2][0]]

In [31]:
video_files = sorted(Path("videos").glob("*.mp4"), key=os.path.getmtime)
input_path = str(video_files[-1])

# NEW OUTPUT FILE
video_path = Path(input_path)

output_dir = Path("cropped_videos")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / f"{video_path.stem}_cropped.mp4"

cap = cv2.VideoCapture(input_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

start_y = int(height * 3 / 5)
new_height = height - start_y

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, new_height)
)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cropped = frame[start_y:height, 0:width]
    out.write(cropped)

cap.release()
out.release()

print(f"Cropped video saved to: {output_path}")

Cropped video saved to: cropped_videos/Final 1 - 2025 Rocket City Regional_cropped.mp4


In [32]:
# RUN
video_files = sorted(Path("cropped_videos").glob("*.mp4"), key=os.path.getmtime)
VIDEO_PATH = str(video_files[-1])

output_dir = REPO_ROOT / "tracking_output"
output_dir.mkdir(exist_ok=True)

botsort_results = run_full_pipeline( 
    VIDEO_PATH,
    ROBOT_MODEL_PATH,
    NUMBER_MODEL_PATH,
    CUSTOM_TRACKER_PATH,
    BLUE_IDS,
    RED_IDS,
    output_path=output_dir / "final_output.mp4",
    tracking_stride=3,
    ocr_stride=15
)


Running BotSort...
Running BotSort tracking...
Tracking complete.
Reading numbers...
Frame 105: []
Frame 120: []
Frame 135: []
Frame 150: []
Frame 165: []
Frame 180: []
Frame 195: []
Frame 210: []
Frame 225: []
Frame 270: []
Frame 285: [{'detected': '233', 'matched': 2338, 'track_id': 6, 'box': [179.73020935058594, 21.692583084106445, 372.2258605957031, 408.3415222167969], 'alliance_hint': 'blue'}]
Frame 300: [{'detected': '23', 'matched': None, 'track_id': 6, 'box': [182.50384521484375, 0.0, 370.2440490722656, 404.9820861816406], 'alliance_hint': 'blue'}, {'detected': '186', 'matched': None, 'track_id': 8, 'box': [1486.586181640625, 2.0824570655822754, 1687.389404296875, 430.9613952636719], 'alliance_hint': 'blue'}]
Frame 315: [{'detected': '23', 'matched': None, 'track_id': 6, 'box': [181.10899353027344, 5.934088230133057, 367.550537109375, 407.6919250488281], 'alliance_hint': 'blue'}]
Frame 330: [{'detected': '8778', 'matched': None, 'track_id': 6, 'box': [170.5452423095703, 87.8438